# 🔍 Fase 5 y 6: Interpretabilidad con SHAP y Respuestas de Negocio
## Proyecto de Ciencia de Datos - Inside Airbnb NYC

---
### 🎯 Objetivos:
1. Explicabilidad global y local del mejor modelo de Machine Learning mediante **SHAP** (*SHapley Additive exPlanations*).
2. Cuantificar el impacto marginal de las comodidades (*amenities*) y la ubicación sobre el precio de mercado.
3. Responder explícitamente a las preguntas de negocio formuladas al inicio del proyecto.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from sklearn.model_selection import train_test_split

from src.models import get_feature_lists, build_preprocessor
import xgboost as xgb

# Carga de datos procesados
data_path = PROJECT_ROOT / "data" / "selected" / "airbnb" / "processed" / "nyc_listings_features.parquet"
df = pd.read_parquet(data_path)

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

num_cols, cat_cols = get_feature_lists()
num_cols = [c for c in num_cols if c in df.columns]
cat_cols = [c for c in cat_cols if c in df.columns]

preprocessor = build_preprocessor(num_cols, cat_cols)
X_train_trans = preprocessor.fit_transform(train_df)
X_test_trans = preprocessor.transform(test_df)

# Obtener nombres de las características post-OneHot
cat_encoder = preprocessor.named_transformers_['cat'].named_steps['onehot']
cat_encoded_names = cat_encoder.get_feature_names_out(cat_cols).tolist()
all_feature_names = num_cols + cat_encoded_names

# Entrenar modelo final XGBoost
model = xgb.XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6, random_state=42)
model.fit(X_train_trans, train_df['log_price'].values)
print("✅ Modelo final entrenado.")

---
## 1. Explicabilidad Global con SHAP Summary Plot
Calculamos los valores SHAP sobre una muestra del conjunto de prueba para identificar las variables más determinantes del precio.

In [ ]:
# Muestra representativa para cálculo de SHAP
sample_size = 1000
X_sample = X_test_trans[:sample_size]

explainer = shap.TreeExplainer(model)
shap_values = explainer(X_sample)
shap_values.feature_names = all_feature_names

# Beeswarm Summary Plot
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, max_display=20, show=False)
plt.title("SHAP Beeswarm Plot - Importancia y Dirección de Variables en el Precio", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 2. Respuestas Cuantitativas a las Preguntas de Negocio

### 📌 Pregunta 1: ¿Qué comodidades tienen mayor retorno marginal sobre el precio por noche?
Analizamos los valores SHAP medios para las variables `has_*`.

In [ ]:
# Extraer importancia SHAP para comodidades
amenity_feature_indices = [i for i, name in enumerate(all_feature_names) if name.startswith('has_')]
amenity_importance = []

for idx in amenity_feature_indices:
    feat_name = all_feature_names[idx]
    mean_abs_shap = np.abs(shap_values.values[:, idx]).mean()
    # Impacto porcentual aproximado en precio: exp(shap) - 1
    amenity_importance.append({
        'Comodidad': feat_name.replace('has_', '').replace('_', ' ').title(),
        'SHAP_Medio': mean_abs_shap
    })

amenity_df_shap = pd.DataFrame(amenity_importance).sort_values(by='SHAP_Medio', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=amenity_df_shap.head(15), x='SHAP_Medio', y='Comodidad', hue='Comodidad', palette='magma', legend=False)
plt.title('Top Comodidades con Mayor Impacto en el Precio (SHAP)', fontsize=13, fontweight='bold')
plt.xlabel('Magnitud de Impacto Medio |SHAP| (escala log)')
plt.ylabel('Comodidad')
plt.tight_layout()
plt.show()

---
### 📌 Pregunta 2: ¿Cómo penaliza la distancia al centro de Manhattan el precio del alojamiento?
Visualizamos el gráfico de dependencia parcial para la distancia a Times Square.

In [ ]:
times_square_idx = all_feature_names.index('dist_times_square_km')

plt.figure(figsize=(10, 5))
shap.dependence_plot(
    times_square_idx,
    shap_values.values,
    X_sample,
    feature_names=all_feature_names,
    interaction_index="room_type_Entire home/apt",
    show=False
)
plt.title("Impacto de la Distancia a Times Square en el Precio según Tipo de Alojamiento", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()